# Uncertainty Analysis: Roll Angle
This notebook analyzes the impact of camera roll angle variations on slope estimation accuracy.
It includes:
1.  **Image Transformation**: Generating perspective views with varying Roll angles (simulated via Phi parameter).
2.  **Result Analysis**: Comparing estimated slopes against ground truth.
3.  **Visualization**: Plotting uncertainty and data availability.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from zensvi.transform import ImageTransformer

# Plotting settings
plt.rcParams.update({'font.size': 14})
plt.rcParams['font.family'] = 'Arial'

In [ ]:
# Configuration & Paths
BASE_DIR = '.'

# Input Data
VALIDATION_SAMPLES_PATH = os.path.join(BASE_DIR, 'subsamples_result.csv')
SOURCE_PANOS_DIR = os.path.join(BASE_DIR, 'pano')

# Working Directories
DEGRADATION_DIR = os.path.join(BASE_DIR, 'roll')
TRANSFORMED_DIR = os.path.join(DEGRADATION_DIR, 'roll-panos')
RESULTS_DIR = os.path.join(DEGRADATION_DIR, 'roll-results')

# Output
FIGURE_SAVE_PATH = os.path.join(BASE_DIR, 'rolled_uncertainty.png')

# Parameters
ROLL_ANGLES = list(range(-30, 31, 5))

In [ ]:
# Load validation samples
subsamples = pd.read_csv(VALIDATION_SAMPLES_PATH)
display(subsamples.head())

In [ ]:
def run_roll_transformation(input_dir, base_output_dir, angles):
    """
    Transform panoramas with varying Roll angles.
    Note: This uses the 'phi' parameter to simulate the angle change as per the original analysis.
    """
    for angle in angles:
        output_dir = os.path.join(base_output_dir, f'chunk_{angle}_rolled')
        print(f"Processing Roll {angle}° -> {output_dir}")
        
        image_transformer = ImageTransformer(dir_input=input_dir, dir_output=output_dir)
        image_transformer.transform_images(
            style_list="perspective",
            FOV=90,
            theta=90,
            phi=angle,  
            aspects=(10, 10),
            show_size=100
        )
        
        # Cleanup
        os.system(f"find {output_dir} -name '*Direction_0*' -delete")
        os.system(f"find {output_dir} -name '*Direction_180*' -delete")

run_roll_transformation(SOURCE_PANOS_DIR, TRANSFORMED_DIR, ROLL_ANGLES)

In [ ]:
## run vision2slope model on transformed images
## Import Vision2Slope modules
from vision2slope import (
    PipelineConfig,
    Vision2SlopePipeline
)

# iterate over rolled directories and process images (TRANSFORMED_DIR)

for angle in ROLL_ANGLES:
    input_dir = os.path.join(TRANSFORMED_DIR, f'chunk_{angle}_rolled', 'perspective')
    output_dir = os.path.join(RESULTS_DIR, f'chunk_{angle}_rolled')
    print(f"Running Vision2Slope on Roll {angle}° images")

    config = PipelineConfig(
        input_dir=input_dir,           # Input image directory
        output_dir=output_dir    # Output results directory
    )
    pipeline = Vision2SlopePipeline(config)
    results = pipeline.process_batch()

In [ ]:
def search_csv_files(directory, keyword=None):
    csv_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.csv'):
                if keyword is None or keyword in file:
                    csv_files.append(os.path.join(root, file))
    return csv_files

def load_roll_results(base_path, angles, sample_df):
    """Load and merge results from different Roll runs."""
    merged_df = sample_df[['pano_id', 'slope_1m']].copy()
    merged_df = merged_df.set_index('pano_id')
    
    available_ratios = []
    
    for angle in angles:
        # Note: Folder naming convention from original notebook: chunk_{angle}Rolled
        search_path = os.path.join(base_path, f'chunk_{angle}_rolled')
        csv_files = search_csv_files(search_path, keyword='estimate')
        
        if csv_files:
            # Use the first found CSV
            res_df = pd.read_csv(csv_files[0])
            
            # Calculate availability
            n = len(res_df)
            available_ratios.append(n / 2000 * 100) # Assuming 2000 total samples
            
            res_df.set_index('pano_id', inplace=True)
            res_df = res_df[['road_estimated_slope']]
            res_df = res_df.rename(columns={'road_estimated_slope': f'rolled_{angle}_estimated_slope'})
            
            # Drop duplicates
            res_df = res_df[~res_df.index.duplicated(keep='first')]
            
            merged_df = merged_df.join(res_df, how='left')
            
            # Calculate difference
            merged_df[f'rolled_{angle}_estimated_slope_diff'] = merged_df[f'rolled_{angle}_estimated_slope'] - merged_df['slope_1m']
        else:
            print(f"No results found for Roll {angle}°")
            available_ratios.append(0)
            
    return merged_df, available_ratios

# Run analysis
analyzed_df, availability = load_roll_results(RESULTS_DIR, ROLL_ANGLES, subsamples)

In [ ]:
def plot_roll_summary(sample_df, availability_pct, angles, save_path=None):
    import matplotlib.ticker as mticker
    
    angles = list(angles)
    plot_angles = []
    rename_map = {}
    for angle in angles:
        col = f'rolled_{angle}_estimated_slope_diff'
        if col in sample_df.columns:
            plot_angles.append(angle)
            rename_map[col] = f"{angle}°"

    if not plot_angles:
        print("No data to plot.")
        return

    order_labels = [f"{angle}°" for angle in plot_angles]
    plot_df = (
        sample_df[[f'rolled_{angle}_estimated_slope_diff' for angle in plot_angles]]
        .rename(columns=rename_map)
        .melt(var_name='Roll angle (°)', value_name='Slope difference (°)')
    )
    plot_df['Roll angle (°)'] = pd.Categorical(plot_df['Roll angle (°)'], categories=order_labels, ordered=True)

    # Create figure
    fig = plt.figure(constrained_layout=False, figsize=(12, 8))
    gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.15)
    
    # Boxplot
    highlight_color = sns.color_palette('crest', n_colors=7)[-2]
    ax_box = fig.add_subplot(gs[0])
    sns.boxplot(
        data=plot_df,
        x='Roll angle (°)',
        y='Slope difference (°)',
        ax=ax_box,
        width=0.3,
        fliersize=2.2,
        linewidth=1.2,
        medianprops={'color': '#0F3460', 'linewidth': 2},
        whiskerprops={'color': '#0F3460', 'linewidth': 1.2},
        capprops={'color': '#0F3460', 'linewidth': 1.2},
        showfliers=False,
        showcaps=False
    )
    
    for patch in ax_box.artists:
        patch.set_alpha(0.38)
        patch.set_edgecolor('#0F3460')
        patch.set_facecolor('none')

    ax_box.axhline(0, color='#5B6270', linestyle='--', linewidth=1, alpha=0.8, zorder=0)
    ax_box.set_xlabel('')
    ax_box.set_ylabel('Slope difference (°)', labelpad=12)
    ax_box.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.4)
    ax_box.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}'))
    
    # Remove x-axis labels for boxplot
    ax_box.set_xticklabels([])
    ax_box.tick_params(axis='x', which='both', length=0, labelbottom=False)

    # Availability Line Plot
    positions = np.arange(len(order_labels))
    
    ax_line = fig.add_subplot(gs[1], sharex=ax_box)
    ax_line.plot(positions, availability_pct, color=highlight_color, marker='o', linewidth=2.4, label='Availability')
    ax_line.fill_between(positions, availability_pct, color=highlight_color, alpha=0.15)
    
    ax_line.set_ylabel('Available imagery (%)', labelpad=12)
    ax_line.set_xlabel('Roll angle (°)', labelpad=12)
    ax_line.set_xticks(positions)
    ax_line.set_xticklabels(order_labels)
    ax_line.set_ylim(min(min(availability_pct), 60) * 0.7, max(105, max(availability_pct) * 1.05))
    ax_line.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.35)

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

# Plot
plot_roll_summary(
    analyzed_df,
    availability,
    ROLL_ANGLES,
    save_path=FIGURE_SAVE_PATH
)